In [41]:
!pip install great-expectations pandas -q

In [42]:
from google.colab import files
uploaded = files.upload()

Saving customer_data.csv to customer_data (2).csv


In [43]:
import pandas as pd

df = pd.read_csv("customer_data.csv")
print(df.shape)
print(df.head())

(5015, 7)
  customer_id    age                 email    salary    country         phone  \
0      C04875  999.0  user4875@example.com   41017.0        USA    3637929158   
1      C01026   70.0  user1026@example.com   -1000.0  Australia         -8437   
2      C03546   79.0  user3546@example.com   82797.0        USA  423.366.4508   
3      C03822   20.0           @domain.com   54624.0      India  719-808-4765   
4      C04395   68.0  user4395@example.com  155942.0        USA           NaN   

  signup_date  
0   9/22/2023  
1   2/25/2023  
2   9/15/2023  
3    8/6/2023  
4   5/28/2023  


In [44]:
import re
import os

def load_csv(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    df = pd.read_csv(filepath)
    if df.empty:
        raise ValueError("File is empty")
    return df

def clean_phone(phone):
    if phone is None:
        return ""
    digits = re.sub(r"\D", "", str(phone))
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    return digits if len(digits) == 10 else ""

def validate_email(email):
    if not email or not isinstance(email, str):
        return False
    pattern = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, email.strip()))

# Quick test
print(clean_phone("(318) 414-9221"))
print(validate_email("user@example.com"))
print(validate_email("@domain.com"))

3184149221
True
False


In [45]:
import great_expectations as gx

context  = gx.get_context()
ds       = context.data_sources.add_or_update_pandas(name="customer_ds")
asset    = ds.add_dataframe_asset(name="customer_asset")
batch_def = asset.add_batch_definition_whole_dataframe("customer_batch")

print("GE setup done!")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpktla4p7h' for ephemeral docs site


GE setup done!


In [46]:
from great_expectations.expectations.expectation_configuration import ExpectationConfiguration

suite = context.suites.add(gx.ExpectationSuite(name="customer_data_expectations"))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_not_be_null", kwargs={"column": "customer_id"}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_be_unique", kwargs={"column": "customer_id"}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_be_between", kwargs={"column": "age", "min_value": 0, "max_value": 120}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_match_regex", kwargs={"column": "email", "regex": r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_not_be_null", kwargs={"column": "salary", "mostly": 0.95}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_be_in_set", kwargs={"column": "country", "value_set": ["USA", "Canada", "UK", "Australia"]}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_column_values_to_match_strftime_format", kwargs={"column": "signup_date", "strftime_format": "%m/%d/%Y"}))

suite.add_expectation_configuration(ExpectationConfiguration(
    type="expect_table_row_count_to_be_between", kwargs={"min_value": 500, "max_value": 1000}))

suite.save()
print("8 expectations created!")

8 expectations created!


In [50]:
import re, os, pandas as pd
import great_expectations as gx
from great_expectations.expectations.expectation_configuration import ExpectationConfiguration

df        = pd.read_csv("customer_data.csv")
context   = gx.get_context()
ds        = context.data_sources.add_or_update_pandas(name="ds3")
asset     = ds.add_dataframe_asset(name="asset3")
batch_def = asset.add_batch_definition_whole_dataframe("batch3")
suite     = context.suites.add(gx.ExpectationSuite(name="suite3"))

suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_not_be_null", kwargs={"column": "customer_id"}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_be_unique", kwargs={"column": "customer_id"}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_be_between", kwargs={"column": "age", "min_value": 0, "max_value": 120}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_match_regex", kwargs={"column": "email", "regex": r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_not_be_null", kwargs={"column": "salary", "mostly": 0.95}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_be_in_set", kwargs={"column": "country", "value_set": ["USA","Canada","UK","Australia"]}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_column_values_to_match_strftime_format", kwargs={"column": "signup_date", "strftime_format": "%m/%d/%Y"}))
suite.add_expectation_configuration(ExpectationConfiguration(type="expect_table_row_count_to_be_between", kwargs={"min_value": 500, "max_value": 1000}))

suite.save()
print("Suite ready!")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmppwj9j157' for ephemeral docs site


Suite ready!


In [51]:
vd = context.validation_definitions.add(
    gx.ValidationDefinition(name="vd3", data=batch_def, suite=suite))

results = vd.run(batch_parameters={"dataframe": df})

passed = sum(1 for r in results.results if r.success)

print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
for r in results.results:
    status = "PASS" if r.success else "FAIL"
    col    = r.expectation_config.kwargs.get("column", "TABLE")
    print(f"{status} - {col} - {r.expectation_config.type}")
print("=" * 60)
print(f"Total: {len(results.results)} | Passed: {passed} | Failed: {len(results.results)-passed}")

Calculating Metrics:   0%|          | 0/49 [00:00<?, ?it/s]

VALIDATION SUMMARY
FAIL - customer_id - expect_column_values_to_not_be_null
FAIL - customer_id - expect_column_values_to_be_unique
FAIL - age - expect_column_values_to_be_between
FAIL - email - expect_column_values_to_match_regex
FAIL - salary - expect_column_values_to_not_be_null
FAIL - country - expect_column_values_to_be_in_set
FAIL - signup_date - expect_column_values_to_match_strftime_format
FAIL - TABLE - expect_table_row_count_to_be_between
Total: 8 | Passed: 0 | Failed: 8


In [52]:
context.build_data_docs()

docs_sites = context.get_docs_sites_urls()
for site in docs_sites:
    print("Data Docs path:", site["site_url"])

Data Docs path: file:///tmp/tmppwj9j157/index.html


In [53]:
with open("/tmp/tmpt8cugn0p/index.html", "r") as f:
    html_content = f.read()

from IPython.display import HTML
HTML(html_content)

In [55]:
import shutil
import os

# Remove old folder and copy fresh
if os.path.exists("/content/data_docs"):
    shutil.rmtree("/content/data_docs")

shutil.copytree("/tmp/tmppwj9j157", "/content/data_docs")

print("Data Docs saved!")

Data Docs saved!


In [56]:
%%writefile test_data_utils.py

import re
import os
import pytest
import pandas as pd

# ── paste functions here so pytest can find them ──
def load_csv(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    df = pd.read_csv(filepath)
    if df.empty:
        raise ValueError("File is empty")
    return df

def clean_phone(phone):
    if phone is None:
        return ""
    digits = re.sub(r"\D", "", str(phone))
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    return digits if len(digits) == 10 else ""

def validate_email(email):
    if not email or not isinstance(email, str):
        return False
    pattern = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, email.strip()))


# ── load_csv tests ────────────────────────────────
def test_load_csv_success(tmp_path):
    f = tmp_path / "test.csv"
    f.write_text("id,age\n1,25\n2,30\n")
    df = load_csv(str(f))
    assert len(df) == 2

def test_load_csv_file_not_found():
    with pytest.raises(FileNotFoundError):
        load_csv("missing.csv")

def test_load_csv_empty_file(tmp_path):
    f = tmp_path / "empty.csv"
    f.write_text("id,age\n")
    with pytest.raises(ValueError):
        load_csv(str(f))


# ── clean_phone tests ─────────────────────────────
def test_clean_phone_dashes():
    assert clean_phone("719-808-4765") == "7198084765"

def test_clean_phone_dots():
    assert clean_phone("970.521.2218") == "9705212218"

def test_clean_phone_parentheses():
    assert clean_phone("(318) 414-9221") == "3184149221"

def test_clean_phone_plain():
    assert clean_phone("3637929158") == "3637929158"

def test_clean_phone_none():
    assert clean_phone(None) == ""

def test_clean_phone_invalid():
    assert clean_phone("-8437") == ""


# ── validate_email tests ──────────────────────────
def test_validate_email_valid():
    assert validate_email("user@example.com") is True

def test_validate_email_no_at():
    assert validate_email("userdomain.com") is False

def test_validate_email_starts_with_at():
    assert validate_email("@domain.com") is False

def test_validate_email_none():
    assert validate_email(None) is False

def test_validate_email_empty():
    assert validate_email("") is False

Overwriting test_data_utils.py


In [57]:
!pip install pytest -q
!pytest test_data_utils.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.8.15, typeguard-4.5.2, anyio-4.13.0
collected 14 items                                                             

test_data_utils.py::test_load_csv_success PASSED                         [  7%]
test_data_utils.py::test_load_csv_file_not_found PASSED                  [ 14%]
test_data_utils.py::test_load_csv_empty_file PASSED                      [ 21%]
test_data_utils.py::test_clean_phone_dashes PASSED                       [ 28%]
test_data_utils.py::test_clean_phone_dots PASSED                         [ 35%]
test_data_utils.py::test_clean_phone_parentheses PASSED                  [ 42%]
test_data_utils.py::test_clean_phone_plain PASSED                        [ 50%]
test_data_utils.py::test_clean_phone_none PASSED                         [ 57%]
test_data_utils.py

In [60]:
%%writefile assignment2_report.md

# Assignment 2 Report

# Student Information

**Name:** Eric Rathod

# Objective

The objective of this assignment was to validate the customer dataset with Great Expectations and test utility functions with Pytest. The data set has been purposely made dirty in order to test the validation.

## Validation Results

| Column | Expectation | Result |
|---------|-------------|--------|
| customer_id | Values should not be null | FAIL |
| customer_id | Values should be unique | FAIL |
| age | Values should be between 0 and 120 | FAIL |
| email | Valid email format | FAIL |
| salary | At least 95% values should be present | FAIL |
| country | USA, Canada, UK or Australia | FAIL |
| signup_date | MM/DD/YYYY format | FAIL |
| Table | Row count between 500 and 1000 | FAIL |

### Validation Summary

- Total Expectations: **8**
- Passed: **0**
- Failed: **8**

## Data Quality Issues

- Missing customer IDs: **150**
- Duplicate customer IDs: **452**
- Invalid ages: **384**
- Invalid email addresses: **784**
- Missing salary values: **425**
- Invalid countries: **301**
- Missing signup dates: **14**

## Pytest Results

The following utility functions were tested:

- load_csv()
- clean_phone()
- validate_email()

### Test Summary

- Total Tests: **14**
- Passed: **14**
- Failed: **0**

All test cases passed successfully.

## Great Expectations Data Docs

HTML-formatted Data Docs have been created and contain a visual representation of the results of data validation.

## Reflection

This task shows how important it is to validate data prior to any analysis or machine learning models training. The dataset has some problems including duplicate customer IDs, missing values, wrong email addresses, wrong age values, and wrong country codes.

The most problematic issue is that of duplicate customer IDs as it may lead to duplicates in the dataset and thus to data leakage from training to testing datasets.

Overwriting assignment2_report.md


In [59]:
from google.colab import files
files.download("/content/data_docs/index.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
from google.colab import files
files.download("test_data_utils.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
!zip -r data_docs.zip data_docs

  adding: data_docs/ (stored 0%)
  adding: data_docs/expectations/ (stored 0%)
  adding: data_docs/expectations/suite3.html (deflated 79%)
  adding: data_docs/.ipynb_checkpoints/ (stored 0%)
  adding: data_docs/index.html (deflated 72%)
  adding: data_docs/validations/ (stored 0%)
  adding: data_docs/validations/suite3/ (stored 0%)
  adding: data_docs/validations/suite3/__none__/ (stored 0%)
  adding: data_docs/validations/suite3/__none__/20260627T020915.768987Z/ (stored 0%)
  adding: data_docs/validations/suite3/__none__/20260627T020915.768987Z/ds3-asset3.html (deflated 87%)
  adding: data_docs/static/ (stored 0%)
  adding: data_docs/static/styles/ (stored 0%)
  adding: data_docs/static/styles/data_docs_default_styles.css (deflated 64%)
  adding: data_docs/static/styles/data_docs_custom_styles_template.css (deflated 74%)
  adding: data_docs/static/images/ (stored 0%)
  adding: data_docs/static/images/short-logo-vector.svg (deflated 66%)
  adding: data_docs/static/images/logo-long.png 

In [64]:
from google.colab import files
files.download("data_docs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>